## House Price Prediction With MLFLOW

In this tutorials we will

1.Run a Hyperparameter tuning while training a model

2.Log every Hyperparameter and metrics in the MLFLOW UI

3.Compare the results of the various runs in the MLFLOW UI

4.Choose the best run and register it as a model

In [24]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing)

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

In [3]:
## Preparing the dataset
data=pd.DataFrame(housing.data,columns=housing.feature_names)
data["Price"]=housing.target
data.head(10)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
5,4.0368,52.0,4.761658,1.103627,413.0,2.139896,37.85,-122.25,2.697
6,3.6591,52.0,4.931907,0.951362,1094.0,2.128405,37.84,-122.25,2.992
7,3.1200,52.0,4.797527,1.061824,1157.0,1.788253,37.84,-122.25,2.414
8,2.0804,42.0,4.294118,1.117647,1206.0,2.026891,37.84,-122.26,2.267
9,3.6912,52.0,4.970588,0.990196,1551.0,2.172269,37.84,-122.25,2.611


### Train Test Split, Model Hyperparameter Tuning, MLFLOW Experiments

In this section, we will:
1. **Split the data** - Divide our data into training (80%) and testing (20%) sets
2. **Perform hyperparameter tuning** - Test different model settings to find the best ones
3. **Track experiments with MLFlow** - Log all our experiments so we can compare results later

In [ ]:
from urllib.parse import urlparse

# Separate the features (X) and target (y)
# X = all features like location, bedrooms, bathrooms, etc.
# y = the price we want to predict (target variable)
X = data.drop(columns=["Price"])  # All columns except Price
y = data["Price"]  # Only the Price column (what we want to predict)

In [ ]:
## Hyperparameter tuning using GridSearchCV

# This function will test all combinations of hyperparameters and find the best one
def hyperparameter_tuning(X_train, y_train, param_grid):
    rf = RandomForestRegressor()  # Create a Random Forest model (ensemble of decision trees)
    
    # GridSearchCV tests all parameter combinations with 3-fold cross-validation
    # cv=3 means it splits data into 3 parts and tests on each part
    # n_jobs=-1 uses all CPU cores to speed up the process
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=3,
        n_jobs=-1,
        verbose=2,  # Print progress (2 = very verbose)
        scoring="neg_mean_squared_error"  # Lower scores are better
    )
    
    grid_search.fit(X_train, y_train)  # Start the hyperparameter tuning process
    return grid_search  # Return the object containing best parameters and model

#### What is Hyperparameter Tuning?
Hyperparameters are settings we choose before training our model. Think of them like the recipe ingredients:
- **n_estimators**: How many trees to use in the forest (more trees = more accurate but slower)
- **max_depth**: How deep each tree can grow (deeper = memorizes more but might overfit)
- **min_samples_split**: Minimum samples needed to split a node
- **min_samples_leaf**: Minimum samples needed at a leaf node

GridSearchCV tries ALL combinations of these settings and finds which one works best!

In [23]:
## Step 1: Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)
# We use 80% for training and 20% for testing
# This helps us check if our model can predict on data it hasn't seen before

# Create a signature - this tells MLFlow what kind of data our model expects
from mlflow.models import infer_signature
signature = infer_signature(X_train, y_train)

## Step 2: Define the hyperparameters to test
param_grid = {
    "n_estimators": [100, 200],  # Try 100 and 200 trees
    "max_depth": [5, 10, None],  # Try depths of 5, 10, or unlimited
    "min_samples_split": [2, 5],  # Try 2 or 5 minimum samples to split a node
    "min_samples_leaf": [1, 2]    # Try 1 or 2 minimum samples in a leaf
}
# This creates 2 × 3 × 2 × 2 = 24 combinations to test!

## Step 3: Start MLFlow experiment and train the model
with mlflow.start_run():  # Start recording this experiment
    
    # Perform hyperparameter tuning (tests all 24 combinations)
    grid_search = hyperparameter_tuning(X_train, y_train, param_grid)
    
    # Get the best model (the one with best performance)
    best_model = grid_search.best_estimator_

    # Test the best model on our test data
    y_pred = best_model.predict(X_test)  # Get predictions on test data
    mse = mean_squared_error(y_test, y_pred)  # Calculate error (lower is better)

    ## Step 4: Log the best hyperparameters to MLFlow
    # This saves the settings we used so we can remember them later
    mlflow.log_param("best_n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("best_max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_["min_samples_split"])
    mlflow.log_param("best_min_samples_leaf", grid_search.best_params_["min_samples_leaf"])
    
    ## Step 5: Log the performance metrics
    mlflow.log_metric("best_mse", mse)  # MSE = Mean Squared Error (lower is better)

    ## Step 6: Set up MLFlow tracking to show results in the web interface
    mlflow.set_tracking_uri("http://127.0.0.1:5000")  # Connect to MLFlow server
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    ## Step 7: Save and register the model
    if tracking_url_type_store != "file":
        # If using a remote server, register the model with a name
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="Best Randomforest Model")
    else:
        # If using local files, save the model with signature
        mlflow.sklearn.log_model(best_model, "model", signature=signature)

    ## Step 8: Print results so we can see them
    print(f"Best Hyperparameters: {grid_search.best_params_}")  # Show which settings worked best
    print(f"Mean Squared Error: {mse}")  # Show how accurate our model is

Fitting 3 folds for each of 24 candidates, totalling 72 fits


2026/03/17 01:25:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 01:25:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Best Randomforest Model' already exists. Creating a new version of this model...
2026/03/17 01:26:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best Randomforest Model, version 2
Created version '2' of model 'Best Randomforest Model'.


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Mean Squared Error: 0.263043156020438
🏃 View run intrigued-shrew-634 at: http://127.0.0.1:5000/#/experiments/0/runs/9790e5fa517b4d91967b7a5e41701257
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


#### About MLFlow - Experiment Tracking

**MLFlow** is like a lab notebook for machine learning. It records:
- **Parameters**: The settings we used (n_estimators, max_depth, etc.)
- **Metrics**: How well the model performed (accuracy, error rate, etc.)
- **Models**: Saves the trained model so we can use it later
- **Artifacts**: Any other files we want to keep

This helps us:

✓ Compare different experiments easily

✓ Reproduce results later

✓ Track which experiment performed best

✓ Deploy the best model to production